In [1]:
# ============================================================
# BLOCK 1 — Install Dependencies
# ============================================================
!pip install ultralytics transformers pycocotools nltk rouge-score --quiet
!pip install evaluate sacrebleu --quiet
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
print("All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


All dependencies installed.


In [2]:
# ============================================================
# BLOCK 2 — Imports & Global Config
# ============================================================
import os, json, math, random, time
from collections import Counter
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from PIL import Image
import requests
from io import BytesIO
from transformers import ViTModel, ViTImageProcessor
from ultralytics import YOLO
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

# ── Device setup (T4×2) ───────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS    = torch.cuda.device_count()
print(f"Using {NUM_GPUS} GPU(s): {[torch.cuda.get_device_name(i) for i in range(NUM_GPUS)]}")

# ── Paths (Kaggle structure) ──────────────────────────────────
COCO_ROOT   = Path("/kaggle/input/coco-2017-dataset/coco2017")
TRAIN_IMG   = COCO_ROOT / "train2017"
VAL_IMG     = COCO_ROOT / "val2017"
TRAIN_ANN   = COCO_ROOT / "annotations/captions_train2017.json"
VAL_ANN     = COCO_ROOT / "annotations/captions_val2017.json"
OUTPUT_DIR  = Path("/kaggle/working/fulcrum_captioning")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────
CFG = {
    "vocab_size"      : 10000,
    "max_len"         : 52,       # max caption token length (incl. <SOS>/<EOS>)
    "d_model"         : 768,      # must match ViT hidden size
    "n_heads"         : 8,
    "n_dec_layers"    : 4,
    "dim_feedforward" : 2048,
    "dropout"         : 0.1,
    "embed_dim"       : 512,      # word embedding dim (projected to d_model)
    "batch_size"      : 64,       # per-GPU; effective = 64 × NUM_GPUS
    "epochs"          : 3,
    "lr"              : 3e-4,
    "warmup_steps"    : 4000,
    "grad_clip"       : 5.0,
    "label_smoothing" : 0.1,
    "vit_unfreeze_layers": 2,     # unfreeze last N ViT transformer blocks
}
print("Config ready.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using 2 GPU(s): ['Tesla T4', 'Tesla T4']
Config ready.


In [3]:
import os
ANN_DIR = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations"
print(os.listdir(ANN_DIR))

['person_keypoints_train2017.json', 'instances_val2017.json', 'instances_train2017.json', 'person_keypoints_val2017.json', 'captions_train2017.json', 'captions_val2017.json']


In [4]:
TRAIN_ANN = Path(ANN_DIR) / "captions_train2017.json" 
VAL_ANN   = Path(ANN_DIR) / "captions_val2017.json"

In [5]:
# ============================================================
# BLOCK 3 — Vocabulary (FINAL FIXED)
# ============================================================
import re
import json
from collections import Counter
from pathlib import Path

class Vocabulary:
    PAD, SOS, EOS, UNK = 0, 1, 2, 3
    SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
    def __init__(self):
        self.word2idx = {w: i for i, w in enumerate(self.SPECIAL)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}

    # ── Tokenizer (fast + replaceable) ────────────────────────
    @staticmethod
    def tokenize(text: str):
        text = text.lower()
        text = re.sub(r"[^a-z0-9 ]+", "", text)
        return text.split()

    def build(self, captions: list[str], max_size: int):
        counter = Counter()
        for cap in captions:
            tokens = self.tokenize(cap)
            counter.update(tokens)
        most_common = counter.most_common(max_size - len(self.SPECIAL))
        for word, _ in most_common:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx]  = word
        print(f"  Vocabulary size: {len(self.word2idx)}")

    def encode(self, caption: str, max_len: int):
        tokens = [self.SOS]
        for w in self.tokenize(caption):
            tokens.append(self.word2idx.get(w, self.UNK))
        # Ensure EOS always included
        tokens = tokens[:max_len - 1]
        tokens.append(self.EOS)
        # Pad
        tokens += [self.PAD] * (max_len - len(tokens))
        return tokens

    def decode(self, indices):
        # Handle tensor input
        if hasattr(indices, "tolist"):
            indices = indices.tolist()
        words = []
        for idx in indices:
            if idx == self.EOS:
                break
            if idx not in (self.PAD, self.SOS):
                words.append(self.idx2word.get(idx, "<UNK>"))
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

def build_vocab_from_coco(ann_path: Path, max_size: int) -> Vocabulary:
    # Safety check
    if not ann_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {ann_path}")
    with open(ann_path, "r") as f:
        data = json.load(f)
    if "annotations" not in data:
        raise ValueError("Invalid COCO annotation format")
    captions = [ann["caption"] for ann in data["annotations"]]
    vocab = Vocabulary()
    vocab.build(captions, max_size)
    return vocab

# ── Build + Save ─────────────────────────────────────────────
print("Building vocabulary...")
print(f"Using annotation file: {TRAIN_ANN}")  # debug visibility
vocab = build_vocab_from_coco(TRAIN_ANN, CFG["vocab_size"])
# Safe save (no pickle issues)
torch.save({
    "word2idx": vocab.word2idx,
    "idx2word": vocab.idx2word
}, OUTPUT_DIR / "vocab.pth")
print("Vocabulary built and saved.")

Building vocabulary...
Using annotation file: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_train2017.json
  Vocabulary size: 10000
Vocabulary built and saved.


In [6]:
# ============================================================
# BLOCK 4 — COCO Captions Dataset (FINAL - CORRECT PATHS)
# ============================================================
from torchvision import transforms
from PIL import Image, ImageFile

# Prevent PIL crashes (Ultralytics patch compatibility)
ImageFile.LOAD_TRUNCATED_IMAGES = True

VIT_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    ),
])


class COCOCaptionDataset(Dataset):
    def __init__(self, img_dir: Path, ann_path: Path, vocab: Vocabulary, max_len: int, transform=None):
        with open(ann_path) as f:
            data = json.load(f)
        self.img_dir   = img_dir
        self.vocab     = vocab
        self.max_len   = max_len
        self.transform = transform or VIT_TRANSFORM
        # id → filename mapping
        id2file = {img["id"]: img["file_name"] for img in data["images"]}
        # Filter valid samples ONLY
        self.samples = []
        missing = 0
        for ann in data["annotations"]:
            img_id = ann["image_id"]
            if img_id not in id2file:
                continue
            fname = id2file[img_id]
            img_path = self.img_dir / fname
            if img_path.exists():
                self.samples.append((fname, ann["caption"]))
            else:
                missing += 1
        print(f"  Dataset: {len(self.samples)} valid pairs from {img_dir}")
        print(f"  Skipped {missing} missing images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, caption = self.samples[idx]
        img_path = self.img_dir / fname
        with Image.open(img_path) as img:
            img = img.convert("RGB")
        img_tensor = self.transform(img)
        tokens = self.vocab.encode(caption, self.max_len)
        cap_tensor = torch.tensor(tokens, dtype=torch.long)
        return img_tensor, cap_tensor

# ============================================================
# PATHS
# ============================================================
COCO_ROOT = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017")
TRAIN_IMG = COCO_ROOT / "train2017"
VAL_IMG   = COCO_ROOT / "val2017"
TRAIN_ANN = COCO_ROOT / "annotations/captions_train2017.json"
VAL_ANN   = COCO_ROOT / "annotations/captions_val2017.json"

# ============================================================
# LOAD DATASETS
# ============================================================
print("Loading datasets...")
train_ds = COCOCaptionDataset(TRAIN_IMG, TRAIN_ANN, vocab, CFG["max_len"])
val_ds   = COCOCaptionDataset(VAL_IMG,   VAL_ANN,   vocab, CFG["max_len"])

# ============================================================
# DATALOADERS (DataParallel OPTIMIZED)
# ============================================================
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"] * max(1, NUM_GPUS),
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"] * max(1, NUM_GPUS),
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Loading datasets...
  Dataset: 591753 valid pairs from /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017
  Skipped 0 missing images
  Dataset: 25014 valid pairs from /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017
  Skipped 0 missing images
Train batches: 4623 | Val batches: 196


In [7]:
# ============================================================
# BLOCK 5 — Model Definition (FINAL)
# ============================================================
# ── Positional Encoding ───────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

# ── ViT Encoder Wrapper ───────────────────────────────────────
class ViTEncoder(nn.Module):
    """
    HuggingFace ViT encoder
    Output: (B, 196, 768)  — CLS token removed
    """
    def __init__(self, unfreeze_layers: int = 2):
        super().__init__()
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        # Freeze everything
        for p in self.vit.parameters():
            p.requires_grad = False
        # Unfreeze last N blocks
        total_blocks = len(self.vit.encoder.layer)
        for blk in self.vit.encoder.layer[total_blocks - unfreeze_layers:]:
            for p in blk.parameters():
                p.requires_grad = True
        # Always train final layernorm
        for p in self.vit.layernorm.parameters():
            p.requires_grad = True

    def forward(self, pixel_values):
        out = self.vit(pixel_values=pixel_values)
        return out.last_hidden_state[:, 1:, :]   
        
# ── Transformer Caption Decoder ───────────────────────────────
class CaptionDecoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int,
                 n_layers: int, dim_ff: int, max_len: int, dropout: float):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=Vocabulary.PAD)
        self.embed_dropout = nn.Dropout(dropout)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.scale   = math.sqrt(d_model)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True   # Pre-LN (stable)
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=n_layers)
        self.fc_out  = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, tgt_tokens, memory, tgt_key_padding_mask=None):
        T = tgt_tokens.size(1)
        # causal mask
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(T).to(tgt_tokens.device)
        tgt_mask = tgt_mask.bool()   # 🔥 FIX
        x = self.embed(tgt_tokens) * self.scale
        x = self.embed_dropout(x)
        x = self.pos_enc(x)
        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=None   # no padding in ViT tokens
        )

        return self.fc_out(out)

# ── Full Captioning Model ─────────────────────────────────────
class FULCRUMCaptioner(nn.Module):
    def __init__(self, cfg: dict, vocab_size: int):
        super().__init__()
        self.encoder = ViTEncoder(unfreeze_layers=cfg["vit_unfreeze_layers"])
        self.enc_norm = nn.LayerNorm(cfg["d_model"])  
        self.decoder = CaptionDecoder(
            vocab_size=vocab_size,
            d_model=cfg["d_model"],
            n_heads=cfg["n_heads"],
            n_layers=cfg["n_dec_layers"],
            dim_ff=cfg["dim_feedforward"],
            max_len=cfg["max_len"],
            dropout=cfg["dropout"],
        )

    def forward(self, images, captions):
        """
        images   : (B, 3, 224, 224)
        captions : (B, T)
        """
        memory = self.encoder(images)     # (B, 196, 768)
        memory = self.enc_norm(memory)
        # Teacher forcing
        tgt_in = captions[:, :-1]
        pad_mask = (tgt_in == Vocabulary.PAD)

        logits = self.decoder(
            tgt_tokens=tgt_in,
            memory=memory,
            tgt_key_padding_mask=pad_mask
        )
        return logits   # (B, T-1, vocab_size)

# ── Instantiate Model ─────────────────────────────────────────
model = FULCRUMCaptioner(CFG, len(vocab))
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"  DataParallel across {NUM_GPUS} GPUs.")
model = model.to(DEVICE)

# ── Parameter Stats ───────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready | Total params: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M")

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  DataParallel across 2 GPUs.
Model ready | Total params: 133.3M | Trainable: 61.1M


In [8]:
# ============================================================
# BLOCK 6 — Training 
# ============================================================
criterion = nn.CrossEntropyLoss(
    ignore_index=Vocabulary.PAD,
    label_smoothing=CFG["label_smoothing"]
)

enc = model.module.encoder if NUM_GPUS > 1 else model.encoder
dec = model.module.decoder if NUM_GPUS > 1 else model.decoder
enc_params = [p for p in enc.parameters() if p.requires_grad]
dec_params = list(dec.parameters())

optimizer = torch.optim.AdamW([
    {"params": enc_params, "lr": CFG["lr"] * 0.1},
    {"params": dec_params, "lr": CFG["lr"]},
], weight_decay=1e-4)

def lr_lambda(step):
    step = max(step, 1)
    if step < CFG["warmup_steps"]:
        return step / CFG["warmup_steps"]
    total_steps = len(train_loader) * CFG["epochs"]
    progress = (step - CFG["warmup_steps"]) / max(1, total_steps - CFG["warmup_steps"])
    return max(0.1, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler()

# ============================================================
# TRAIN FUNCTION
# ============================================================
def train_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss, n_batches = 0.0, 0
    t0 = time.time()
    for i, (images, captions) in enumerate(loader):
        images   = images.to(DEVICE, non_blocking=True)
        captions = captions.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits  = model(images, captions)
            targets = captions[:, 1:]
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )
        # NaN protection
        if not torch.isfinite(loss):
            print("Skipping NaN batch")
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            CFG["grad_clip"]
        )
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        n_batches  += 1
        if (i + 1) % 200 == 0:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch} | Step {i+1}/{len(loader)} | "
                  f"Loss: {total_loss/n_batches:.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.2e} | "
                  f"Time: {elapsed:.0f}s")
    return total_loss / max(n_batches, 1)

# ============================================================
# VALIDATION
# ============================================================
def validate(model, loader):
    model.eval()
    total_loss, n_batches = 0.0, 0
    with torch.no_grad():
        for images, captions in loader:
            images   = images.to(DEVICE, non_blocking=True)
            captions = captions.to(DEVICE, non_blocking=True)
            logits  = model(images, captions)
            targets = captions[:, 1:]
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )
            total_loss += loss.item()
            n_batches  += 1
    return total_loss / max(n_batches, 1)

# ============================================================
# MAIN TRAINING LOOP
# ============================================================
best_val_loss = float("inf")
history = []
patience = 3
no_improve_epochs = 0

print("Starting training...")

for epoch in range(1, CFG["epochs"] + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)
    val_loss   = validate(model, val_loader)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    # Early Stopping + Checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_epochs = 0
        ckpt = model.module.state_dict() if NUM_GPUS > 1 else model.state_dict()
        torch.save(ckpt, OUTPUT_DIR / "best_captioner.pth")
        print(f"  Saved best checkpoint (val_loss={val_loss:.4f})")
    else:
        no_improve_epochs += 1
        print(f"  No improvement ({no_improve_epochs}/{patience})")
    if no_improve_epochs >= patience:
        print("Early stopping triggered.")
        break

# Save history
with open(OUTPUT_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2)
print("Training complete.")

Starting training...


/tmp/ipykernel_24/4264289365.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_24/4264289365.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_24/4264289365.py:61: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  Epoch 1 | Step 200/4623 | Loss: 37.3623 | LR: 1.50e-06 | Time: 128s
  Epoch 1 | Step 400/4623 | Loss: 29.6499 | LR: 3.00e-06 | Time: 251s
  Epoch 1 | Step 600/4623 | Loss: 25.1701 | LR: 4.50e-06 | Time: 371s
  Epoch 1 | Step 800/4623 | Loss: 22.3261 | LR: 6.00e-06 | Time: 487s
  Epoch 1 | Step 1000/4623 | Loss: 20.3187 | LR: 7.50e-06 | Time: 600s
  Epoch 1 | Step 1200/4623 | Loss: 18.7855 | LR: 9.00e-06 | Time: 712s
  Epoch 1 | Step 1400/4623 | Loss: 17.5544 | LR: 1.05e-05 | Time: 822s
  Epoch 1 | Step 1600/4623 | Loss: 16.5288 | LR: 1.20e-05 | Time: 928s
  Epoch 1 | Step 1800/4623 | Loss: 15.6504 | LR: 1.35e-05 | Time: 1031s
  Epoch 1 | Step 2000/4623 | Loss: 14.8823 | LR: 1.50e-05 | Time: 1137s
  Epoch 1 | Step 2200/4623 | Loss: 14.1944 | LR: 1.65e-05 | Time: 1239s
  Epoch 1 | Step 2400/4623 | Loss: 13.5858 | LR: 1.80e-05 | Time: 1341s
  Epoch 1 | Step 2600/4623 | Loss: 13.0488 | LR: 1.95e-05 | Time: 1443s
  Epoch 1 | Step 2800/4623 | Loss: 12.5690 | LR: 2.10e-05 | Time: 1546s
  Ep

In [9]:
# ============================================================
# BLOCK 7 — YOLOv8 Object Detection (FIXED)
# ============================================================
yolo = YOLO("yolov8n.pt").to(DEVICE)  

def detect_objects(image_path: str, conf_threshold: float = 0.3) -> dict:
    """
    Runs YOLOv8 on an image.
    Returns:
      {
        "labels": [...],
        "boxes": [...],
        "scores": [...],
        "summary": "person, laptop, ..."
      }
    """
    # faster than predict()
    results = yolo(image_path, conf=conf_threshold, verbose=False)[0]
    labels, boxes, scores = [], [], []
    for box in results.boxes:
        cls_id = int(box.cls.item())
        labels.append(results.names[cls_id])
        boxes.append(box.xyxy[0].cpu().tolist())
        scores.append(round(box.conf.item(), 2))
    # Deduplicate labels
    seen = dict.fromkeys(labels)
    summary = ", ".join(seen.keys()) if seen else "no objects detected"
    return {
        "labels": labels,
        "boxes": boxes,
        "scores": scores,
        "summary": summary
    }

# ── Quick sanity test ─────────────────────────────────────────
test_img = str(next(VAL_IMG.iterdir()))
det = detect_objects(test_img)
print(f"Detected: {det['summary']}")
print("YOLO stage ready.")

Detected: car, person, bicycle, traffic light, fire hydrant
YOLO stage ready.


In [10]:
# ============================================================
# BLOCK 8 — Evaluation Metrics (FINAL FIXED)
# ============================================================
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

def greedy_decode(model, image_tensor, vocab, max_len, device):
    model.eval()
    enc_model = model.module if isinstance(model, nn.DataParallel) else model
    with torch.no_grad():
        img = image_tensor.unsqueeze(0).to(device)
        # FIX: include encoder normalization
        mem = enc_model.enc_norm(enc_model.encoder(img))
        token_ids = [Vocabulary.SOS]
        for _ in range(max_len - 1):
            tgt = torch.tensor([token_ids], dtype=torch.long, device=device)
            pad_mask = (tgt == Vocabulary.PAD)
            logits = enc_model.decoder(
                tgt, mem,
                tgt_key_padding_mask=pad_mask
            )
            next_id = logits[0, -1].argmax().item()
            token_ids.append(next_id)
            if next_id == Vocabulary.EOS:
                break
    return vocab.decode(token_ids[1:])

def beam_search_decode(model, image_tensor, vocab, max_len, device, beam_width=5):
    enc_model = model.module if isinstance(model, nn.DataParallel) else model
    model.eval()
    with torch.no_grad():
        img = image_tensor.unsqueeze(0).to(device)
        # FIX: encoder normalization
        mem = enc_model.enc_norm(enc_model.encoder(img))
        beams = [(0.0, [Vocabulary.SOS])]
        completed = []
        for _ in range(max_len - 1):
            candidates = []
            for log_prob, tokens in beams:
                if tokens[-1] == Vocabulary.EOS:
                    completed.append((log_prob, tokens))
                    continue
                tgt = torch.tensor([tokens], dtype=torch.long, device=device)
                pad_mask = (tgt == Vocabulary.PAD)
                logits = enc_model.decoder(
                    tgt, mem,
                    tgt_key_padding_mask=pad_mask
                )
                lp = F.log_softmax(logits[0, -1], dim=-1)
                top_lp, top_ids = lp.topk(beam_width)
                for lp_val, tok_id in zip(top_lp.tolist(), top_ids.tolist()):
                    candidates.append((log_prob + lp_val, tokens + [tok_id]))
            if not candidates:
                break
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]
        completed += beams
        best = max(completed, key=lambda x: x[0] / max(len(x[1]), 1))
    return vocab.decode(best[1][1:])

def evaluate_bleu_meteor(model, loader, vocab, device, n_batches=50):
    refs_all, hyps_all = [], []
    model.eval()
    for i, (images, captions) in enumerate(loader):
        if i % 5 == 0:
            print(f"Evaluating batch {i}/{n_batches}")
        if i >= n_batches:
            break
        for img_t, cap_t in zip(images, captions):
            hyp = greedy_decode(model, img_t, vocab, CFG["max_len"], device)
            ref = vocab.decode(cap_t.tolist())
            hyps_all.append(hyp.split())
            refs_all.append([ref.split()])
    smooth = SmoothingFunction().method1
    bleu1 = corpus_bleu(refs_all, hyps_all,
                        weights=(1, 0, 0, 0),
                        smoothing_function=smooth)
    bleu4 = corpus_bleu(refs_all, hyps_all,
                        weights=(0.25, 0.25, 0.25, 0.25),
                        smoothing_function=smooth)
    meteor_scores = [
        meteor_score(r, h)
        for r, h in zip(refs_all, hyps_all)
    ]
    avg_meteor = np.mean(meteor_scores)
    print(f"BLEU-1: {bleu1:.4f} | BLEU-4: {bleu4:.4f} | METEOR: {avg_meteor:.4f}")
    return {
        "bleu1": bleu1,
        "bleu4": bleu4,
        "meteor": avg_meteor
    }

# ── Run evaluation ────────────────────────────────────────────
print("Running evaluation (first 50 val batches)...")
metrics = evaluate_bleu_meteor(model, val_loader, vocab, DEVICE)

Running evaluation (first 50 val batches)...
Evaluating batch 0/50
Evaluating batch 5/50
Evaluating batch 10/50
Evaluating batch 15/50
Evaluating batch 20/50
Evaluating batch 25/50
Evaluating batch 30/50
Evaluating batch 35/50
Evaluating batch 40/50
Evaluating batch 45/50
Evaluating batch 50/50
BLEU-1: 0.3664 | BLEU-4: 0.0837 | METEOR: 0.2758
